# BERT Document Classification — M4 MacBook Optimized 🚀

**Optimized for Apple Silicon M4 10-core MacBook with Authenticator.ai cleaned dataset**

This notebook trains BERT for document classification with M4-specific optimizations:

- **4 Categories**: Education, Insurance, Legal, Resume/Employment (2,047 text samples)
- **Apple Silicon MPS**: Optimized for M4 GPU acceleration
- **Memory Efficient**: Batch size and gradient accumulation optimized for M4
- **Performance Tuned**: 10-core CPU utilization, optimized data loading
- **MVP Ready**: Direct integration with your cleaned dataset splits
- **Deployment Ready**: Model export and inference functions included

## M4 Optimizations:
- MPS (Metal Performance Shaders) acceleration
- Gradient accumulation for effective large batch training
- Memory-efficient data loading with optimal worker count
- Early stopping and learning rate scheduling


In [ ]:

# Install M4-optimized packages
%pip install torch torchvision torchaudio --quiet
%pip install transformers datasets scikit-learn evaluate accelerate --quiet
%pip install psutil matplotlib seaborn --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, json, random, time, warnings
import numpy as np
import pandas as pd
from collections import Counter
import psutil

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

from transformers import (
    AutoTokenizer, AutoModel, AutoConfig, 
    get_linear_schedule_with_warmup,
    logging as hf_logging
)

import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')
hf_logging.set_verbosity_error()

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# M4-optimized device setup
def setup_m4_device():
    if torch.backends.mps.is_available():
        device = 'mps'
        print("🚀 Using Apple Silicon MPS acceleration")
        torch.backends.mps.empty_cache()
    elif torch.cuda.is_available():
        device = 'cuda'
        print("🚀 Using CUDA acceleration")
    else:
        device = 'cpu'
        print("💻 Using CPU")
    return device

DEVICE = setup_m4_device()

# System info
cpu_count = psutil.cpu_count()
memory = psutil.virtual_memory()
print(f"🖥️  M4 System: {cpu_count} cores, {memory.total // (1024**3)} GB RAM")
print(f"📱 PyTorch: {torch.__version__}, MPS: {torch.backends.mps.is_available()}")
print(f"🎯 Device: {DEVICE}")

DEVICE

/Users/prathamsaurabh/Authenticator.ai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'cpu'

## Config
Choose **one** data-loading mode below and set column names.

In [ ]:
# M4-Optimized Configuration for Authenticator.ai MVP
class M4Config:
    # ---- Data Paths (Using your cleaned dataset) ----
    TRAIN_PATH = '../data/training_data/model_splits/text/train.csv'
    VAL_PATH = '../data/training_data/model_splits/text/val.csv'
    TEST_PATH = '../data/training_data/model_splits/text/test.csv'
    
    # ---- Column Names (Your cleaned data format) ----
    TEXT_COL = 'text'
    CATEGORY_COL = 'category'  # Main categories: education, insurance, legal, resume_employment
    SUBTYPE_COL = 'subtype'    # Specific document types
    
    # ---- M4-Optimized Model & Training ----
    MODEL_NAME = 'bert-base-uncased'
    MAX_LEN = 256              # Optimal for M4 memory
    BATCH_SIZE = 8             # Small batch for M4 efficiency
    GRADIENT_ACCUMULATION = 4  # Effective batch size = 8 * 4 = 32
    LR = 2e-5
    EPOCHS = 5                 # More epochs for better convergence
    WARMUP_RATIO = 0.1
    PATIENCE = 3
    WEIGHT_DECAY = 0.01
    
    # ---- M4 Performance Settings ----
    NUM_WORKERS = 4            # Optimized for M4 10-core
    PIN_MEMORY = True
    PREFETCH_FACTOR = 2
    GRADIENT_CLIP = 1.0
    
    # ---- Output ----
    OUTPUT_DIR = 'artifacts_m4_mvp'
    
config = M4Config()

print("🔧 M4-Optimized Configuration:")
print(f"  📊 Dataset: {config.TRAIN_PATH}")
print(f"  🎯 Categories: {config.CATEGORY_COL}")
print(f"  📝 Text Column: {config.TEXT_COL}")
print(f"  🔢 Batch Size: {config.BATCH_SIZE} (x{config.GRADIENT_ACCUMULATION} = {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION})")
print(f"  🧠 Max Length: {config.MAX_LEN}")
print(f"  ⚡ Workers: {config.NUM_WORKERS}")
print(f"  📱 Device: {DEVICE}")

os.makedirs(config.OUTPUT_DIR, exist_ok=True)


## Load Data
This block handles both **single CSV** and **three CSV** workflows. It also aligns column names and optionally enforces real-only evaluation.

In [ ]:
# Load Authenticator.ai cleaned dataset
print("📊 Loading Authenticator.ai cleaned dataset...")

# Load the pre-split data
train_df = pd.read_csv(config.TRAIN_PATH)
val_df = pd.read_csv(config.VAL_PATH)
test_df = pd.read_csv(config.TEST_PATH)

# Clean and prepare text data
for df in [train_df, val_df, test_df]:
    df[config.TEXT_COL] = df[config.TEXT_COL].fillna('').astype(str)
    df[config.CATEGORY_COL] = df[config.CATEGORY_COL].astype(str)
    df[config.SUBTYPE_COL] = df[config.SUBTYPE_COL].astype(str)

print(f"📈 Dataset Statistics:")
print(f"  Train: {len(train_df)} samples")
print(f"  Validation: {len(val_df)} samples") 
print(f"  Test: {len(test_df)} samples")
print(f"  Total: {len(train_df) + len(val_df) + len(test_df)} samples")

# Analyze categories
print(f"\n🏷️  Category Distribution:")
category_counts = train_df[config.CATEGORY_COL].value_counts()
print(category_counts)

print(f"\n📋 Document Subtypes:")
subtype_counts = train_df[config.SUBTYPE_COL].value_counts()
print(f"Total subtypes: {len(subtype_counts)}")
print(subtype_counts.head(10))

# Encode categories for classification (MVP focuses on main categories)
category_encoder = LabelEncoder()
train_df['label'] = category_encoder.fit_transform(train_df[config.CATEGORY_COL])
val_df['label'] = category_encoder.transform(val_df[config.CATEGORY_COL])
test_df['label'] = category_encoder.transform(test_df[config.CATEGORY_COL])

# Save label mapping
label_map = {int(i): cls for i, cls in enumerate(category_encoder.classes_)}
with open(f'{config.OUTPUT_DIR}/category_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)

num_labels = len(category_encoder.classes_)
print(f"\n🎯 MVP Classification:")
print(f"Categories: {list(category_encoder.classes_)}")
print(f"Number of labels: {num_labels}")

print(f"\n📄 Sample Data:")
print(train_df[[config.TEXT_COL, config.CATEGORY_COL, config.SUBTYPE_COL, 'label']].head(3))

train_df.head()


Train: 51706 | Val: 2901 | Test: 2902 | num_labels=1


/var/folders/tf/_g_w7ph97cb_jcz6s2cpxt6c0000gn/T/ipykernel_27522/193829588.py:38: DtypeWarning: Columns (5,13,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = standardize_columns(pd.read_csv(TRAIN_PATH), TEXT_COL, LABEL_COL, SOURCE_COL)


,text,label_raw,source_type,label
0,sample text content for image_classification i...,Other,real,0
1,sample text content for image_classification i...,Other,real,0
2,sample text content for image_classification i...,Other,real,0
3,sample text content for image_classification i...,Other,real,0
4,sample text content for image_classification i...,Other,real,0


In [ ]:
## Tokenizer & Dataset

In [ ]:
# M4-Optimized Tokenizer & Dataset
print("🔤 Loading M4-optimized tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

class M4OptimizedTextDataset(Dataset):
    """M4-optimized dataset with memory efficiency"""
    
    def __init__(self, df, tokenizer, max_len=256, text_col='text'):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.text_col = text_col
        
        # Clean text data
        self.df[text_col] = self.df[text_col].fillna('').astype(str)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row[self.text_col])[:1000]  # Truncate very long texts for M4 efficiency
        
        # Tokenize with M4 optimizations
        enc = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(int(row['label']), dtype=torch.long)
        }

# Create M4-optimized datasets
train_ds = M4OptimizedTextDataset(train_df, tokenizer, config.MAX_LEN, config.TEXT_COL)
val_ds = M4OptimizedTextDataset(val_df, tokenizer, config.MAX_LEN, config.TEXT_COL)
test_ds = M4OptimizedTextDataset(test_df, tokenizer, config.MAX_LEN, config.TEXT_COL)

# M4-optimized data loaders with performance tuning
train_loader = DataLoader(
    train_ds, 
    batch_size=config.BATCH_SIZE, 
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
    prefetch_factor=config.PREFETCH_FACTOR,
    persistent_workers=True
)

val_loader = DataLoader(
    val_ds, 
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
    prefetch_factor=config.PREFETCH_FACTOR,
    persistent_workers=True
)

test_loader = DataLoader(
    test_ds, 
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
    prefetch_factor=config.PREFETCH_FACTOR,
    persistent_workers=True
)

print(f"✅ M4-Optimized Datasets Created:")
print(f"  Train: {len(train_ds)} samples ({len(train_loader)} batches)")
print(f"  Val: {len(val_ds)} samples ({len(val_loader)} batches)")
print(f"  Test: {len(test_ds)} samples ({len(test_loader)} batches)")
print(f"  Workers: {config.NUM_WORKERS}, Pin Memory: {config.PIN_MEMORY}")

len(train_ds), len(val_ds), len(test_ds)


(51706, 2901, 2902)

## Model (BERT encoder + classifier)

In [ ]:
# M4-Optimized BERT Model
class M4BertClassifier(nn.Module):
    """M4-optimized BERT classifier with memory efficiency"""
    
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        
        # Load BERT with M4 memory optimizations
        self.config = AutoConfig.from_pretrained(
            model_name, 
            output_hidden_states=False,
            output_attentions=False  # Disable attention outputs for memory
        )
        
        self.bert = AutoModel.from_pretrained(model_name, config=self.config)
        
        # Classification head with improved initialization
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        
        # Initialize classifier weights for better convergence
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)
    
    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        # M4-optimized forward pass
        outputs = self.bert(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            token_type_ids=token_type_ids
        )
        
        # Use CLS token for classification
        cls_output = outputs.last_hidden_state[:, 0]  # CLS token
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
            return {'loss': loss, 'logits': logits}
        
        return {'logits': logits}

# Initialize M4-optimized model
print("🤖 Initializing M4-optimized BERT model...")
model = M4BertClassifier(config.MODEL_NAME, num_labels).to(DEVICE)

# Model statistics
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"📊 M4 Model Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / (1024**2):.1f} MB")
print(f"  Device: {next(model.parameters()).device}")
print(f"  Categories: {num_labels}")

model


BertClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

## Optimizer, Scheduler, Class Weights

In [ ]:
# M4-Optimized Training Setup
print("⚙️ Setting up M4-optimized training...")

# Calculate class weights for balanced training
cnt = Counter(train_df['label'])
total = sum(cnt.values())
class_weights = torch.tensor([total/(num_labels*cnt[i]) for i in range(num_labels)], dtype=torch.float32).to(DEVICE)
print(f'📊 Class distribution: {dict(cnt)}')
print(f'⚖️  Class weights: {class_weights.cpu().numpy()}')

# M4-optimized optimizer with better settings
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=config.LR, 
    weight_decay=config.WEIGHT_DECAY,
    eps=1e-8,  # Better numerical stability
    betas=(0.9, 0.999)  # Optimal for M4
)

# Calculate training steps with gradient accumulation
steps_per_epoch = len(train_loader)
total_training_steps = steps_per_epoch * config.EPOCHS
num_warmup_steps = int(config.WARMUP_RATIO * total_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps, 
    total_training_steps
)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)

print(f"🔧 Training Configuration:")
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total training steps: {total_training_steps}")
print(f"  Warmup steps: {num_warmup_steps}")
print(f"  Gradient accumulation: {config.GRADIENT_ACCUMULATION}")
print(f"  Effective batch size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION}")


Class weights: tensor([1.])


## Train & Validate (Early Stopping on macro-F1)

In [ ]:
# M4-Optimized Training Loop with Progress Tracking
from tqdm.auto import tqdm
import time

def run_epoch_with_progress(dataloader, training=True, epoch_num=1):
    """M4-optimized training/validation with detailed progress tracking"""
    model.train() if training else model.eval()
    
    phase = "Training" if training else "Validation"
    losses, all_preds, all_labels = [], [], []
    
    # Progress bar with detailed info
    pbar = tqdm(dataloader, 
                desc=f"Epoch {epoch_num} - {phase}", 
                leave=True,
                dynamic_ncols=True)
    
    batch_count = 0
    accumulated_loss = 0
    
    # For gradient accumulation
    if training:
        optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(pbar):
        batch_count += 1
        
        # Move batch to device
        input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
        attention_mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)
        
        with torch.set_grad_enabled(training):
            # Forward pass with new model format
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs['loss'] if isinstance(outputs, dict) else loss_fn(outputs, labels)
            
            if training:
                # Scale loss for gradient accumulation
                loss = loss / config.GRADIENT_ACCUMULATION
                loss.backward()
                accumulated_loss += loss.item()
                
                # Gradient accumulation step
                if (batch_idx + 1) % config.GRADIENT_ACCUMULATION == 0 or (batch_idx + 1) == len(dataloader):
                    # Gradient clipping for stability
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    
                    # Update progress with accumulated loss
                    pbar.set_postfix({
                        'Loss': f'{accumulated_loss:.4f}',
                        'Batch': f'{batch_count}/{len(dataloader)}',
                        'LR': f'{scheduler.get_last_lr()[0]:.2e}',
                        'Device': DEVICE
                    })
                    accumulated_loss = 0
            else:
                # Validation mode
                pbar.set_postfix({
                    'Loss': f'{loss.item():.4f}',
                    'Batch': f'{batch_count}/{len(dataloader)}',
                    'Device': DEVICE
                })
        
        # Collect predictions and labels
        with torch.no_grad():
            if isinstance(outputs, dict):
                logits = outputs['logits']
            else:
                logits = outputs
            
            preds = logits.argmax(dim=-1).detach().cpu().numpy()
            lbls = labels.detach().cpu().numpy()
            
            losses.append(loss.item() * (config.GRADIENT_ACCUMULATION if training else 1))
            all_preds.extend(list(preds))
            all_labels.extend(list(lbls))
        
        # Clear cache periodically for M4 memory management
        if batch_count % 50 == 0 and DEVICE == 'mps':
            torch.backends.mps.empty_cache()
    
    # Calculate metrics
    avg_loss = np.mean(losses)
    acc = accuracy_score(all_labels, all_preds)
    f1m = f1_score(all_labels, all_preds, average='macro')
    
    return avg_loss, acc, f1m

# M4-Optimized Training Loop
print("🚀 Starting M4-Optimized BERT Training...")
print(f"📊 Training on {len(train_ds)} samples, validating on {len(val_ds)} samples")
print(f"🎯 Categories: {list(category_encoder.classes_)}")
print("=" * 80)

best_f1, patience_counter = -1.0, 0
training_history = []
start_time = time.time()

for epoch in range(1, config.EPOCHS + 1):
    epoch_start = time.time()
    
    print(f"\n🔄 EPOCH {epoch}/{config.EPOCHS}")
    print(f"⏰ Started at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Training phase
    tr_loss, tr_acc, tr_f1 = run_epoch_with_progress(train_loader, training=True, epoch_num=epoch)
    
    # Validation phase
    va_loss, va_acc, va_f1 = run_epoch_with_progress(val_loader, training=False, epoch_num=epoch)
    
    # Calculate epoch time
    epoch_time = time.time() - epoch_start
    
    # Store history
    epoch_stats = {
        'epoch': epoch,
        'train_loss': tr_loss,
        'val_loss': va_loss,
        'train_acc': tr_acc,
        'val_acc': va_acc,
        'train_f1': tr_f1,
        'val_f1': va_f1,
        'epoch_time': epoch_time
    }
    training_history.append(epoch_stats)
    
    # Print epoch summary
    print(f"\n📈 EPOCH {epoch} SUMMARY:")
    print(f"  🏃 Train: Loss={tr_loss:.4f}, Acc={tr_acc:.4f}, F1={tr_f1:.4f}")
    print(f"  ✅ Val:   Loss={va_loss:.4f}, Acc={va_acc:.4f}, F1={va_f1:.4f}")
    print(f"  ⏱️  Time: {epoch_time:.1f}s")
    
    # Early stopping and model saving
    if va_f1 > best_f1:
        best_f1 = va_f1
        patience_counter = 0
        
        # Save best model
        print(f"  🎉 New best F1: {best_f1:.4f} - Saving model!")
        os.makedirs(config.OUTPUT_DIR, exist_ok=True)
        torch.save(model.state_dict(), f'{config.OUTPUT_DIR}/best_model.pt')
        
        # Save training history
        with open(f'{config.OUTPUT_DIR}/training_history.json', 'w') as f:
            json.dump(training_history, f, indent=2)
    else:
        patience_counter += 1
        print(f"  ⏳ No improvement - Patience: {patience_counter}/{config.PATIENCE}")
        
        if patience_counter >= config.PATIENCE:
            print(f"\n🛑 Early stopping triggered after {epoch} epochs!")
            break

total_time = time.time() - start_time
print(f"\n🏁 TRAINING COMPLETED!")
print(f"  🏆 Best Validation F1: {best_f1:.4f}")
print(f"  ⏱️  Total Time: {total_time/60:.1f} minutes")
print(f"  💾 Model saved to: {config.OUTPUT_DIR}/best_model.pt")


## Test Evaluation

In [ ]:
# M4-Optimized Test Evaluation
print("🧪 Loading best model for test evaluation...")
model.load_state_dict(torch.load(f'{config.OUTPUT_DIR}/best_model.pt', map_location=DEVICE))
model.to(DEVICE)
model.eval()

print("🔍 Running test evaluation with progress tracking...")
all_preds, all_labels = [], []

# Test evaluation with progress bar
test_pbar = tqdm(test_loader, desc="Test Evaluation", leave=True)
for batch in test_pbar:
    input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
    attention_mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
    labels = batch['labels'].to(DEVICE, non_blocking=True)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        if isinstance(outputs, dict):
            logits = outputs['logits']
        else:
            logits = outputs
            
        preds = logits.argmax(dim=-1).detach().cpu().numpy()
        lbls = labels.detach().cpu().numpy()
        
        all_preds.extend(list(preds))
        all_labels.extend(list(lbls))
    
    test_pbar.set_postfix({'Samples': len(all_preds)})

# Calculate final metrics
test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='macro')
test_f1_weighted = f1_score(all_labels, all_preds, average='weighted')

print(f"\n🏆 FINAL TEST RESULTS:")
print(f"  📊 Test Accuracy: {test_acc:.4f}")
print(f"  📈 Test F1 (Macro): {test_f1:.4f}")
print(f"  📈 Test F1 (Weighted): {test_f1_weighted:.4f}")
print(f"  📝 Test Samples: {len(all_labels)}")

print(f"\n📋 DETAILED CLASSIFICATION REPORT:")
print("=" * 60)
target_names = [category_encoder.classes_[i] for i in range(len(category_encoder.classes_))]
print(classification_report(all_labels, all_preds, target_names=target_names, digits=4))

# Save test results
test_results = {
    'test_accuracy': test_acc,
    'test_f1_macro': test_f1,
    'test_f1_weighted': test_f1_weighted,
    'test_samples': len(all_labels),
    'categories': list(category_encoder.classes_),
    'classification_report': classification_report(all_labels, all_preds, target_names=target_names, digits=4, output_dict=True)
}

with open(f'{config.OUTPUT_DIR}/test_results.json', 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"\n💾 Results saved to: {config.OUTPUT_DIR}/test_results.json")


In [ ]:
model.load_state_dict(torch.load('artifacts/model.pt', map_location=DEVICE))
model.to(DEVICE); model.eval()
all_preds, all_labels = [], []
for batch in test_loader:
    input_ids = batch['input_ids'].to(DEVICE)
    attention_mask = batch['attention_mask'].to(DEVICE)
    token_type_ids = batch.get('token_type_ids')
    if token_type_ids is not None: token_type_ids = token_type_ids.to(DEVICE)
    labels = batch['labels'].to(DEVICE)
    with torch.no_grad():
        logits, _ = model(input_ids, attention_mask, token_type_ids, labels)
        preds = logits.argmax(dim=-1).detach().cpu().numpy()
        lbls = labels.detach().cpu().numpy()
        all_preds.extend(list(preds)); all_labels.extend(list(lbls))
acc = accuracy_score(all_labels, all_preds)
f1m = f1_score(all_labels, all_preds, average='macro')
print({'test_accuracy': acc, 'test_f1_macro': f1m})
print('\nPer-class report:\n')
print(classification_report(all_labels, all_preds, digits=4))


## Inference Helper

In [ ]:
import torch.nn.functional as F

def load_label_map(path='artifacts/label_map.json'):
    with open(path) as f:
        return {int(k): v for k,v in json.load(f).items()}

def predict(texts, batch_size=32, return_labels=True):
    model.eval(); label_map = load_label_map()
    preds_all, probs_all = [], []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        enc = tokenizer(chunk, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k,v in enc.items()}
        with torch.no_grad():
            logits = model(**enc)
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=1)
        if return_labels:
            preds_lbl = [label_map[int(p)] for p in preds]
            preds_all.extend(preds_lbl)
        else:
            preds_all.extend(preds.tolist())
        probs_all.extend(probs.tolist())
    return preds_all, probs_all

# Example:
# preds, probs = predict(["This is a sample.", "Another line of text."])
# preds, probs[:1]
